In [ ]:
# import libraries
import numpy as np
import pandas as pd
from pathlib import Path

In [41]:
# set data directory
DATA_DIR = Path("data/sets")

# combine every saved set
pokemon_all = pd.concat(
    [pd.read_csv(path, dtype={"number": str, "product_id": str, "variant": str}, parse_dates=["date"]) for path in DATA_DIR.glob("*.csv")],
    ignore_index=True
)

# 
sets_df = pd.read_csv("data/sets_info.csv", parse_dates=["release_date"])
pokemon_all = pokemon_all.merge(sets_df, on="set_slug", how="left")


In [60]:
# cleaning
pokemon_all["variant"] = pokemon_all["variant"].fillna("Standard")
pokemon_all["price"] = pokemon_all["price"].replace(0, np.nan)
pokemon_all["date"] = pokemon_all["date"].dt.to_period("M").dt.to_timestamp()
pokemon_all["decade"] = (pokemon_all["release_date"].dt.year // 10) * 10



for col in ["grade", "set_slug", "set_name", "language", "name", "variant"]:
    pokemon_all[col] = pokemon_all[col].astype("category")

In [ ]:
pokemon_all.info()

pokemon_all["variant"].unique()
pokemon_all["ungraded"].unique()

<class 'pandas.DataFrame'>
RangeIndex: 13124130 entries, 0 to 13124129
Data columns (total 12 columns):
 #   Column        Dtype         
---  ------        -----         
 0   grade         category      
 1   date          datetime64[us]
 2   price         float64       
 3   set_slug      category      
 4   product_id    str           
 5   name          category      
 6   variant       category      
 7   number        str           
 8   set_name      category      
 9   language      category      
 10  release_date  datetime64[us]
 11  promo         bool          
dtypes: bool(1), category(6), datetime64[us](2), float64(1), str(2)
memory usage: 638.4 MB


['Grade 9.5', 'Grade 7', 'Grade 9', 'PSA 10', 'Grade 8', 'Ungraded']
Categories (6, str): ['Grade 7', 'Grade 8', 'Grade 9', 'Grade 9.5', 'PSA 10', 'Ungraded']

In [57]:
set_comparison = pokemon_all[(pokemon_all["variant"] == "Standard") & (pokemon_all['grade'] == "PSA 10")]

set_comparison["decade"] = (set_comparison["release_date"].dt.year // 10) * 10

set_comparison

,grade,date,price,set_slug,product_id,name,variant,number,set_name,language,release_date,promo,decade
159,PSA 10,2022-06-01,NaN,pokemon-1999-topps-movie,3560849,pikachu's vacation,Standard,42,Pokemon 1999 Topps Movie,English,1999-06-01,False,1990.0
160,PSA 10,2022-07-01,NaN,pokemon-1999-topps-movie,3560849,pikachu's vacation,Standard,42,Pokemon 1999 Topps Movie,English,1999-06-01,False,1990.0
161,PSA 10,2022-08-01,89.05,pokemon-1999-topps-movie,3560849,pikachu's vacation,Standard,42,Pokemon 1999 Topps Movie,English,1999-06-01,False,1990.0
162,PSA 10,2022-09-01,109.97,pokemon-1999-topps-movie,3560849,pikachu's vacation,Standard,42,Pokemon 1999 Topps Movie,English,1999-06-01,False,1990.0
163,PSA 10,2022-10-01,106.74,pokemon-1999-topps-movie,3560849,pikachu's vacation,Standard,42,Pokemon 1999 Topps Movie,English,1999-06-01,False,1990.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
13120095,PSA 10,2026-06-01,32.00,pokemon-xy,806611,great ball,Standard,118,Pokemon XY,English,2014-02-05,False,2010.0
13120096,PSA 10,2026-07-01,32.39,pokemon-xy,806611,great ball,Standard,118,Pokemon XY,English,2014-02-05,False,2010.0
13120097,PSA 10,2026-08-01,31.40,pokemon-xy,806611,great ball,Standard,118,Pokemon XY,English,2014-02-05,False,2010.0
13120098,PSA 10,2026-09-01,31.88,pokemon-xy,806611,great ball,Standard,118,Pokemon XY,English,2014-02-05,False,2010.0


In [59]:
cards_per_set = (
    set_comparison.groupby("set_name", observed=True)["product_id"]
    .nunique()
    .sort_values(ascending=False) 
)

cards_per_set

sets_per_decade = (
    set_comparison.groupby("decade", observed=True)["set_name"]
    .nunique()
    .sort_values(ascending=False)
)

sets_per_decade

decade
2020.0    52
2000.0    50
2010.0    49
1990.0     6
Name: set_name, dtype: int64

In [53]:
price_per_set = (
    set_comparison.groupby("set_name", observed=True)["price"]
    .max()
    .round(2)
    .sort_values(ascending=False) 
)

price_per_set

set_name
Pokemon Promo                          715920.74
Pokemon Call of Legends                238500.00
Pokemon Legends Awakened               186095.05
Pokemon Expedition                     173715.67
Pokemon Rising Rivals                  160922.04
                                         ...    
Pokemon McDonalds 2023                    376.17
Pokemon Trick or Trade 2024               369.39
Pokemon McDonalds 2024                    325.00
Pokemon Trick or Trade 2022               300.00
Pokemon TCG Classic: Blastoise Deck       285.00
Name: price, Length: 158, dtype: float64

In [ ]:
# # ================================
# # BEESWARM — every point visible
# # ================================
# # Works with any data: one row per point with a value and a name (and a group for version 1).
# # Set the column names, swap in your data, and list any points to highlight.
# # Two versions:
# #   1. grouped: one swarm per category, stacked down the page
# #   2. single: one big swarm across the page, with chosen points highlighted and labelled

# # %%
# import matplotlib.pyplot as plt
# import numpy as np
# import pandas as pd

# import chartgazing as cg

# # %% ================================
# # your data
# # ================================
# VALUE, GROUP, NAME = "ratio", "region", "town"

# rng = np.random.default_rng(11)
# regions = {"London": (11, 2.2, 32), "South East": (9.5, 1.8, 45), "North West": (6.2, 1.4, 40),
#            "Scotland": (5.4, 1.2, 38)}
# df = pd.concat(
#     [pd.DataFrame({GROUP: r, VALUE: np.clip(rng.normal(m, s, n), 2, 20)})
#      for r, (m, s, n) in regions.items()],
#     ignore_index=True,
# )
# df[NAME] = [f"Town {i}" for i in range(len(df))]
# df.loc[df[GROUP].eq("North West").idxmax(), [NAME, VALUE]] = ["Manchester", 8.9]

# highlight = ["Manchester"]
# x_range = (0, 18)
# unit_suffix = "x"

# title = "House prices are furthest out of reach in the South"
# subtitle = "House price to earnings ratio, by town"
# note = "Note: illustrative data. Each dot is one town."
# source = "Source: your source."

# marker_size = 4.5      # dot diameter in points


# def swarm(x, diameter):
#     """y offsets (same units as x) so dots of the given diameter don't overlap. Greedy, centre-out."""
#     order = np.argsort(x)
#     ys = np.zeros(len(x))
#     placed_x, placed_y = [], []
#     for i in order:
#         xi = x[i]
#         near = [(xj, yj) for xj, yj in zip(placed_x, placed_y) if abs(xj - xi) < diameter]
#         candidates = [0.0]
#         for xj, yj in near:
#             dy = np.sqrt(diameter ** 2 - (xi - xj) ** 2)
#             candidates += [yj + dy, yj - dy]
#         for c in sorted(candidates, key=abs):
#             if all((xi - xj) ** 2 + (c - yj) ** 2 >= diameter ** 2 - 1e-9 for xj, yj in near):
#                 ys[i] = c
#                 break
#         placed_x.append(xi)
#         placed_y.append(ys[i])
#     return ys


# # %% ================================
# # 1. grouped swarms
# # ================================
# order = df.groupby(GROUP)[VALUE].median().sort_values().index.tolist()   # highest median on top

# fig, ax = cg.chart(title, subtitle, note=note, source=source, grid="x")

# ax.set_xlim(*x_range)
# ax.set_ylim(-0.6, len(order) - 0.4)
# ax.set_yticks(range(len(order)), order)
# cg.number_axis(ax.xaxis, suffix=unit_suffix)

# # the swarm is laid out in points, so fit the layout first to know the final axes size
# cg.fit(fig)
# bbox = ax.get_window_extent()
# pt = 72 / fig.dpi
# x_per_pt = np.diff(ax.get_xlim())[0] / (bbox.width * pt)
# y_per_pt = np.diff(ax.get_ylim())[0] / (bbox.height * pt)

# for yi, group_name in enumerate(order):
#     group = df[df[GROUP] == group_name]
#     offsets = swarm(group[VALUE].to_numpy() / x_per_pt, marker_size * 1.05) * y_per_pt
#     is_hl = group[NAME].isin(highlight).to_numpy()

#     ax.scatter(group[VALUE][~is_hl], yi + offsets[~is_hl], s=marker_size ** 2,
#                color=cg.SECONDARY, alpha=0.75, linewidth=0, clip_on=False)
#     ax.scatter(group[VALUE][is_hl], yi + offsets[is_hl], s=(marker_size * 1.3) ** 2,
#                color=cg.PRIMARY, edgecolor=cg.BACKGROUND, linewidth=0.6, zorder=3, clip_on=False)

#     for name, xv, yo in zip(group[NAME][is_hl], group[VALUE][is_hl], offsets[is_hl]):
#         ax.annotate(name, xy=(xv, yi + yo), xytext=(0, 6), textcoords="offset points",
#                     ha="center", va="bottom", fontsize=7.5, fontweight="bold", color=cg.TEXT)

# cg.save(fig, cg.EXAMPLES / "09_beeswarm.png")
# plt.show()